# 11 — Strategy Mining + Predictive Models (CPU)

This replaces **E003 + E004 + the replay-derived reaction portion of E007**.

## Inputs
- Required: output Dataset from notebook 10
- Code repo Dataset: optional. If absent and Internet is ON, the exact commit recorded by notebook 10 is cloned/checked out.
- Accelerator: **None**
- Internet: OFF if the code repo Dataset is attached; otherwise ON for auto-clone.

## Outputs
- macro strategy archetypes;
- tiny runtime opponent-archetype model;
- 24-turn opponent supply/sell predictor;
- win diagnostic model;
- conditional-reaction archetypes;
- market-threshold research tables;
- one `learned_model.json` ready for offline search.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,shutil

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/kagv2')
WORK.mkdir(parents=True,exist_ok=True)

def find_repo():
    roots=[INPUT,Path('/kaggle/working'),Path.cwd()]
    hit=next((p for r in roots if r.exists() for p in r.rglob('src/kagv2/__init__.py')),None)
    return hit.parents[2] if hit else None

def expected_commit():
    hits=list(INPUT.rglob('repo_commit.txt')) if INPUT.exists() else []
    if hits:
        x=hits[0].read_text().strip()
        return x if x else None
    return None

ROOT=find_repo()
if ROOT is None:
    dst=Path('/kaggle/working/kaggriculture')
    if not dst.exists():
        r=subprocess.run(
            ['git','clone','--depth','1','https://github.com/sidhulyalkar/kaggriculture.git',str(dst)],
            capture_output=True,text=True
        )
        if r.returncode:
            raise RuntimeError(
                'Could not find an attached code repo and GitHub clone failed. '
                'Turn Internet ON or attach your kaggriculture-code-repo Dataset.\n'+r.stderr[-2000:]
            )
    ROOT=dst
    ref=expected_commit()
    if ref:
        subprocess.run(['git','-C',str(ROOT),'fetch','--depth','1','origin',ref],capture_output=True,text=True)
        c=subprocess.run(['git','-C',str(ROOT),'checkout',ref],capture_output=True,text=True)
        if c.returncode:
            print('WARN: could not checkout pinned commit',ref,c.stderr[-500:])

sys.path.insert(0,str(ROOT))
sys.path.insert(0,str(ROOT/'src'))
commit=subprocess.run(['git','-C',str(ROOT),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
if commit:
    (WORK/'repo_commit.txt').write_text(commit)
print('ROOT =',ROOT)
print('WORK =',WORK)
print('COMMIT =',commit or 'dataset snapshot')


In [ ]:
import pandas as pd,numpy as np,json
def find_input(name):
    hits=list(INPUT.rglob(name))
    if not hits: raise FileNotFoundError(f'Attach notebook 10 output; missing {name}')
    return hits[0]

turns=pd.read_parquet(find_input('turns.parquet'))
daily=pd.read_parquet(find_input('daily_macros.parquet'))
bt_path=list(INPUT.rglob('bt_strength.csv'))
bt=pd.read_csv(bt_path[0]) if bt_path else pd.DataFrame()
print('turns',turns.shape,'daily',daily.shape)


In [ ]:
# -------- Macro strategy mining --------
from src.kagv2.macros import episode_profiles,fit_archetypes,build_macro_library,save_json

profiles=episode_profiles(daily)
n_profiles=len(profiles)
# Adaptive cluster count: enough diversity without fragmenting tiny samples.
N_CLUSTERS=max(3,min(8,round(np.sqrt(max(9,n_profiles))/4)))
N_CLUSTERS=min(N_CLUSTERS,max(2,n_profiles)) if n_profiles else 2
print('profiles',n_profiles,'clusters',N_CLUSTERS)

clustered,cluster_model=fit_archetypes(profiles,n_clusters=N_CLUSTERS)
clustered.to_parquet(WORK/'archetype_profiles.parquet',index=False)
lib=build_macro_library(clustered,daily)
save_json(lib,WORK/'macro_library.json')
save_json(cluster_model,WORK/'offline_archetype_model.json')

display(clustered.groupby('archetype').agg(
    episodes=('episode_id','size'),
    mean_reward=('final_reward','mean'),
    win_rate=('win_target','mean')
).sort_values(['win_rate','mean_reward'],ascending=False))


In [ ]:
# -------- Predictive models --------
from sklearn.preprocessing import StandardScaler
from src.kagv2.models import train_win_model,train_supply_model,save_model_bundle
from src.kagv2.features import public_feature_frame
from src.kagv2.constants import PUBLIC_RUNTIME_FEATURES

win,win_metrics=train_win_model(turns)
supply,supply_metrics=train_supply_model(turns,horizon=24,alpha=10.0)

labels=clustered[['episode_id','player','archetype']].drop_duplicates()
d=turns.merge(labels,on=['episode_id','player'],how='inner')
d=d[d.day.between(5,18) & d.hour.isin([0,6,12,18])].copy()
X=public_feature_frame(d)
sc=StandardScaler().fit(X)
Z=sc.transform(X)
centroids=[]
cluster_ids=sorted(d.archetype.unique())
for cl in cluster_ids:
    centroids.append(Z[d.archetype.to_numpy()==cl].mean(0).tolist())

arch={
    'runtime_features':list(X.columns),
    'mean':sc.mean_.tolist(),
    'scale':sc.scale_.tolist(),
    'centroids':centroids,
    'cluster_ids':[int(x) for x in cluster_ids],
    'posterior_temperature':1.0,
}
bundle=save_model_bundle(
    WORK/'learned_model.json',
    win=win,supply=supply,archetype=arch,macro_library=lib
)
metrics={'win':win_metrics,'supply':supply_metrics,'n_archetypes':len(centroids),'n_training_checkpoints':len(d)}
(WORK/'model_metrics.json').write_text(json.dumps(metrics,indent=2))
print(json.dumps(metrics,indent=2))
assert set(supply['features'])==set(PUBLIC_RUNTIME_FEATURES)
assert not any('shed' in f or 'seed_' in f for f in supply['features'])


In [ ]:
# -------- Conditional reactions + threshold research --------
from src.kagv2.replay import add_future_opponent_sell_labels
from src.kagv2.reactions import extract_reaction_events,reaction_profiles,fit_reaction_archetypes
from src.kagv2.probes import threshold_reactivity

turns1=add_future_opponent_sell_labels(turns,horizon=1)
events=extract_reaction_events(turns1,price_threshold=.10,inventory_threshold=30,post_turns=12)
rp=reaction_profiles(events,bt_strength=bt,min_events=3)
events.to_parquet(WORK/'reaction_events.parquet',index=False)
rp.to_parquet(WORK/'reaction_profiles.parquet',index=False)

reaction_summary={'events':len(events),'profiles':len(rp)}
if len(rp)>=4:
    k=min(8,max(2,len(rp)//8))
    rc,rm=fit_reaction_archetypes(rp,n_clusters=k)
    rc.to_parquet(WORK/'reaction_archetypes.parquet',index=False)
    (WORK/'reaction_model.json').write_text(json.dumps(rm,indent=2))
    reaction_summary['clusters']=k

thresholds={}
for p in ['STRAWBERRY','MELON','MILK','WOOL','WHEAT','FERTILIZER']:
    t=threshold_reactivity(turns1,p,bins=24,min_bin=20)
    thresholds[p]=t.to_dict(orient='records')
(WORK/'probe_thresholds.json').write_text(json.dumps(thresholds,indent=2,default=str))
(WORK/'reaction_summary.json').write_text(json.dumps(reaction_summary,indent=2))
print(reaction_summary)


In [ ]:
# -------- Promotion diagnostics --------
# These gates do not prove ladder value. They prevent obviously weak learned
# components from silently becoming the default.
promotion={
    'supply_candidate': bool((supply_metrics.get('r2') or -999) > 0.0),
    'win_model_informative': bool((win_metrics.get('auc') or 0.0) > 0.55),
    'archetype_count':len(centroids),
    'runtime_features':len(PUBLIC_RUNTIME_FEATURES),
}
(WORK/'model_promotion_diagnostics.json').write_text(json.dumps(promotion,indent=2))
print(json.dumps(promotion,indent=2))
print('Next: save this output Dataset and run notebook 12.')
